# Reachy 1.2 Simulator — Phase 0 Smoke Test

Mirrors the Phase 0 hardware smoke test checklist from the project roadmap.  
Run against the simulated SDK server (fake mode) inside this container.

**SDK:** `reachy_sdk` (v1) — NOT `reachy2_sdk`  
**Server:** `fake_reachy_server.py` running on `localhost:50051`

In [11]:
from reachy_sdk import ReachySDK
import time

REACHY_HOST = 'localhost'
REACHY_PORT = 50051   # fake_reachy_server.py; physical robot uses 50055
print(f'Connecting to {REACHY_HOST}:{REACHY_PORT}...')

Connecting to localhost:50051...


## 1. SDK Connection

In [12]:
reachy = ReachySDK(host=REACHY_HOST, sdk_port=REACHY_PORT)
print('✓ Connected to ReachySDK')
print(f'  Host: {REACHY_HOST}:{REACHY_PORT}')

✓ Connected to ReachySDK
  Host: localhost:50051


Exception in callback PollerCompletionQueue._handle_events(<_UnixSelecto...e debug=False>)()
handle: <Handle PollerCompletionQueue._handle_events(<_UnixSelecto...e debug=False>)()>
Traceback (most recent call last):
  File "/usr/lib/python3.8/asyncio/events.py", line 81, in _run
    self._context.run(self._callback, *self._args)
  File "src/python/grpcio/grpc/_cython/_cygrpc/aio/completion_queue.pyx.pxi", line 147, in grpc._cython.cygrpc.PollerCompletionQueue._handle_events
BlockingIOError: [Errno 11] Resource temporarily unavailable
Exception in callback PollerCompletionQueue._handle_events(<_UnixSelecto...e debug=False>)()
handle: <Handle PollerCompletionQueue._handle_events(<_UnixSelecto...e debug=False>)()>
Traceback (most recent call last):
  File "/usr/lib/python3.8/asyncio/events.py", line 81, in _run
    self._context.run(self._callback, *self._args)
  File "src/python/grpcio/grpc/_cython/_cygrpc/aio/completion_queue.pyx.pxi", line 147, in grpc._cython.cygrpc.PollerCompletionQu

## 2. Right Arm Joints
Expected DXL IDs per wiring diagram: 10–17

In [3]:
arm_joints = reachy.r_arm.joints
print(f'✓ Right arm detected: {len(list(arm_joints.values()))} joints')
for name, joint in arm_joints.items():
    print(f'  {name}: present_position={joint.present_position:.1f}°')

✓ Right arm detected: 8 joints
  r_shoulder_pitch: present_position=0.0°
  r_shoulder_roll: present_position=0.0°
  r_arm_yaw: present_position=0.0°
  r_elbow_pitch: present_position=0.0°
  r_forearm_yaw: present_position=0.0°
  r_wrist_pitch: present_position=0.0°
  r_wrist_roll: present_position=0.0°
  r_gripper: present_position=0.0°


## 3. Head Joints
neck_roll, neck_pitch, neck_yaw (Orbita), l_antenna, r_antenna

In [4]:
head_joints = reachy.head.joints
print(f'✓ Head detected: {len(list(head_joints.values()))} joints')
for name, joint in head_joints.items():
    print(f'  {name}: present_position={joint.present_position:.1f}°')

✓ Head detected: 5 joints
  neck_roll: present_position=0.0°
  neck_pitch: present_position=0.0°
  neck_yaw: present_position=0.0°
  l_antenna: present_position=0.0°
  r_antenna: present_position=0.0°


## 4. Arm Power On / Off

In [5]:
reachy.turn_on('r_arm')
print(f'✓ Right arm ON  — r_shoulder_pitch compliant: {reachy.r_arm.r_shoulder_pitch.compliant}')
time.sleep(0.3)
reachy.turn_off('r_arm')
print(f'✓ Right arm OFF — r_shoulder_pitch compliant: {reachy.r_arm.r_shoulder_pitch.compliant}')

✓ Right arm ON  — r_shoulder_pitch compliant: False
✓ Right arm OFF — r_shoulder_pitch compliant: True


✓ Right arm OFF — r_shoulder_pitch compliant: True


## 5. Gripper Open / Close
Gripper is a Dynamixel joint — open = positive angle, close = 0°

In [6]:
reachy.turn_on('r_arm')

# Position arm so the gripper is visible in RViz before opening/closing
reachy.r_arm.r_shoulder_pitch.goal_position = -40.0
reachy.r_arm.r_elbow_pitch.goal_position    = -80.0
reachy.r_arm.r_wrist_pitch.goal_position    =  30.0
print('Arm in pick-ready pose (shoulder -40°, elbow -80°, wrist +30°) — watch RViz...')
time.sleep(2.5)

reachy.r_arm.r_gripper.goal_position = 50.0
print('✓ Gripper opening  (goal → 50°)')
time.sleep(2.0)

reachy.r_arm.r_gripper.goal_position = 0.0
print('✓ Gripper closing  (goal → 0°)')
time.sleep(1.5)

# Return arm to home before next section
for j in reachy.r_arm.joints.values():
    j.goal_position = 0.0
time.sleep(2.0)
reachy.turn_off('r_arm')

Arm in pick-ready pose (shoulder -40°, elbow -80°, wrist +30°) — watch RViz...
✓ Gripper opening  (goal → 50°)
✓ Gripper closing  (goal → 0°)


## 6. Head — look_at, Neck Roll, Antennae
`look_at` uses analytic head IK (neck_roll/pitch/yaw). Neck roll and antennae are
direct joint control. Watch the head pan, tilt, and the antennae wiggle in RViz.

In [ ]:
reachy.turn_on('head')

print('--- look_at sequence ---')
reachy.head.look_at(x=1.0, y=0.0,  z=0.0,  duration=1.0); print('  → forward');     time.sleep(2.0)
reachy.head.look_at(x=1.0, y=0.4,  z=0.0,  duration=1.0); print('  → left');        time.sleep(2.0)
reachy.head.look_at(x=1.0, y=-0.4, z=0.0,  duration=1.0); print('  → right');       time.sleep(2.0)
reachy.head.look_at(x=1.0, y=0.0,  z=-0.35,duration=1.0); print('  → down (table)');time.sleep(2.0)
reachy.head.look_at(x=1.0, y=0.0,  z=0.0,  duration=1.0); print('  → neutral');     time.sleep(1.5)
print('✓ Head look_at (forward / left / right / table / neutral)')

print('\n--- Neck roll (head tilt) ---')
reachy.head.neck_roll.goal_position =  25.0; time.sleep(2.0); print('  → tilt right (+25°)')
reachy.head.neck_roll.goal_position = -25.0; time.sleep(2.0); print('  → tilt left  (-25°)')
reachy.head.neck_roll.goal_position =   0.0; time.sleep(1.5); print('  → upright')
print('✓ Neck roll')

print('\n--- Antennae ---')
reachy.head.l_antenna.goal_position =  60.0
reachy.head.r_antenna.goal_position =  60.0; time.sleep(1.5); print('  → both up (+60°)')
reachy.head.l_antenna.goal_position = -60.0
reachy.head.r_antenna.goal_position = -60.0; time.sleep(1.5); print('  → both down (-60°)')
reachy.head.l_antenna.goal_position =  60.0
reachy.head.r_antenna.goal_position = -60.0; time.sleep(1.5); print('  → split: left up, right down')
reachy.head.l_antenna.goal_position = -60.0
reachy.head.r_antenna.goal_position =  60.0; time.sleep(1.5); print('  → split: left down, right up')
reachy.head.l_antenna.goal_position =   0.0
reachy.head.r_antenna.goal_position =   0.0; time.sleep(1.0); print('  → antennae home')
print('✓ Antennae')

reachy.turn_off('head')

## 7. Arm Motion Sequence
Four distinct poses with 2.5 s holds so you can observe each transition in RViz (`localhost:6080`).  
Pose → reach forward → arm out to side → elbow high → pick-ready → home.

In [10]:
reachy.turn_on('r_arm')
print('Watch RViz at localhost:6080 — each pose holds 2.5 s\n')

def set_pose(label, **joints):
    for name, deg in joints.items():
        getattr(reachy.r_arm, name).goal_position = deg
    goals = '  '.join(f'{n.replace("r_","")}={deg:+.0f}°' for n, deg in joints.items())
    print(f'  → {label}')
    print(f'     {goals}')
    time.sleep(2.5)

# Pose 1 — arm reaching forward and down toward a table surface
set_pose('Reach forward',
    r_shoulder_pitch=-60.0,
    r_elbow_pitch=-70.0,
    r_wrist_pitch=15.0)

# Pose 2 — arm extended out to the right side
set_pose('Arm out to side',
    r_shoulder_pitch=-10.0,
    r_shoulder_roll=-55.0,
    r_elbow_pitch=0.0,
    r_wrist_pitch=0.0)

# Pose 3 — elbow high / bent overhead
set_pose('Elbow high',
    r_shoulder_pitch=-15.0,
    r_shoulder_roll=-20.0,
    r_arm_yaw=30.0,
    r_elbow_pitch=-100.0,
    r_forearm_yaw=20.0)

# Pose 4 — pick-ready: arm forward + down, wrist angled, gripper open
set_pose('Pick-ready (gripper open)',
    r_shoulder_pitch=-45.0,
    r_shoulder_roll=-10.0,
    r_arm_yaw=0.0,
    r_elbow_pitch=-85.0,
    r_forearm_yaw=-10.0,
    r_wrist_pitch=30.0,
    r_gripper=50.0)

# Return home
print('  → Home  (all joints → 0°)')
for j in reachy.r_arm.joints.values():
    j.goal_position = 0.0
time.sleep(2.5)

print(f'\n✓ Arm motion sequence complete — 4 poses demonstrated')
reachy.turn_off('r_arm')

Watch RViz at localhost:6080 — each pose holds 2.5 s

  → Reach forward
     shouldepitch=-60°  elbow_pitch=-70°  wrist_pitch=+15°
  → Arm out to side
     shouldepitch=-10°  shoulderoll=-55°  elbow_pitch=+0°  wrist_pitch=+0°
  → Elbow high
     shouldepitch=-15°  shoulderoll=-20°  arm_yaw=+30°  elbow_pitch=-100°  forearm_yaw=+20°
  → Pick-ready (gripper open)
     shouldepitch=-45°  shoulderoll=-10°  arm_yaw=+0°  elbow_pitch=-85°  forearm_yaw=-10°  wrist_pitch=+30°  gripper=+50°
  → Home  (all joints → 0°)

✓ Arm motion sequence complete — 4 poses demonstrated


## 8. Left Arm Motion Sequence
Mirror of the right arm sequence — all four poses on the left side.
Note: shoulder roll sign is opposite (positive = arm out to the left).

In [ ]:
reachy.turn_on('l_arm')
print('Watch RViz — left arm poses, 2.5 s each\n')

def set_l_pose(label, **joints):
    for name, deg in joints.items():
        getattr(reachy.l_arm, name).goal_position = deg
    goals = '  '.join(f'{n.replace("l_","")}={deg:+.0f}°' for n, deg in joints.items())
    print(f'  → {label}')
    print(f'     {goals}')
    time.sleep(2.5)

# Pose 1 — reach forward (same pitch/elbow convention as right arm)
set_l_pose('Reach forward',
    l_shoulder_pitch=-60.0,
    l_elbow_pitch=-70.0,
    l_wrist_pitch=15.0)

# Pose 2 — arm out to the left (roll sign is positive for left arm)
set_l_pose('Arm out to left side',
    l_shoulder_pitch=-10.0,
    l_shoulder_roll=55.0,
    l_elbow_pitch=0.0,
    l_wrist_pitch=0.0)

# Pose 3 — elbow high
set_l_pose('Elbow high',
    l_shoulder_pitch=-15.0,
    l_shoulder_roll=20.0,
    l_arm_yaw=-30.0,
    l_elbow_pitch=-100.0,
    l_forearm_yaw=-20.0)

# Pose 4 — pick-ready + gripper open
set_l_pose('Pick-ready (gripper open)',
    l_shoulder_pitch=-45.0,
    l_shoulder_roll=10.0,
    l_arm_yaw=0.0,
    l_elbow_pitch=-85.0,
    l_forearm_yaw=10.0,
    l_wrist_pitch=30.0,
    l_gripper=50.0)

print('  → Home  (all joints → 0°)')
for j in reachy.l_arm.joints.values():
    j.goal_position = 0.0
time.sleep(2.5)

print('\n✓ Left arm motion sequence complete')
reachy.turn_off('l_arm')

## 9. Combined — Both Arms + Head Simultaneously
The full robot in motion at once: symmetric reach, forward reach toward the table,
antennae express, and home. This is the closest to what the real tabletop task looks like.

In [ ]:
reachy.turn_on('r_arm')
reachy.turn_on('l_arm')
reachy.turn_on('head')
print('Watch RViz — full robot in motion\n')

# --- 1. Arms open wide, head looks forward ---
reachy.r_arm.r_shoulder_pitch.goal_position = -20.0
reachy.r_arm.r_shoulder_roll.goal_position  = -50.0
reachy.r_arm.r_elbow_pitch.goal_position    = -40.0
reachy.l_arm.l_shoulder_pitch.goal_position = -20.0
reachy.l_arm.l_shoulder_roll.goal_position  =  50.0
reachy.l_arm.l_elbow_pitch.goal_position    = -40.0
reachy.head.look_at(x=1.0, y=0.0, z=0.0, duration=1.0)
print('  → Arms wide open, head forward'); time.sleep(3.0)

# --- 2. Both arms reach forward + head looks down at table ---
reachy.r_arm.r_shoulder_pitch.goal_position = -55.0
reachy.r_arm.r_shoulder_roll.goal_position  =  -8.0
reachy.r_arm.r_elbow_pitch.goal_position    = -75.0
reachy.r_arm.r_wrist_pitch.goal_position    =  20.0
reachy.l_arm.l_shoulder_pitch.goal_position = -55.0
reachy.l_arm.l_shoulder_roll.goal_position  =   8.0
reachy.l_arm.l_elbow_pitch.goal_position    = -75.0
reachy.l_arm.l_wrist_pitch.goal_position    =  20.0
reachy.head.look_at(x=1.0, y=0.0, z=-0.35, duration=1.0)
print('  → Both arms reaching to table, head looks down'); time.sleep(3.0)

# --- 3. Right picks, left steadies, head watches right side ---
reachy.r_arm.r_gripper.goal_position        =  50.0
reachy.l_arm.l_gripper.goal_position        =   0.0
reachy.head.look_at(x=1.0, y=-0.3, z=-0.3, duration=1.0)
print('  → Right gripper opens, head looks at pick point'); time.sleep(2.5)
reachy.r_arm.r_gripper.goal_position        =   0.0
print('  → Right gripper closes (grasp)'); time.sleep(2.0)

# --- 4. Antennae celebrate ---
reachy.head.l_antenna.goal_position =  70.0
reachy.head.r_antenna.goal_position =  70.0
reachy.head.look_at(x=1.0, y=0.0, z=0.0, duration=0.5)
print('  → Antennae up! (task complete expression)'); time.sleep(2.0)
reachy.head.l_antenna.goal_position =   0.0
reachy.head.r_antenna.goal_position =   0.0

# --- 5. Return all to home ---
for j in reachy.r_arm.joints.values(): j.goal_position = 0.0
for j in reachy.l_arm.joints.values(): j.goal_position = 0.0
reachy.head.look_at(x=1.0, y=0.0, z=0.0, duration=1.0)
print('  → Home'); time.sleep(3.0)

print('\n✓ Combined motion complete')
reachy.turn_off('r_arm')
reachy.turn_off('l_arm')
reachy.turn_off('head')

## 10. Results Summary

In [ ]:
checks = [
    'SDK connection to fake gRPC server (port 50051)',
    'Right arm joints readable (8 joints, UIDs 10-17)',
    'Head joints readable (5 joints, UIDs 30-34)',
    'Arm turn_on / turn_off (compliant toggle)',
    'Right gripper open/close with arm in pick-ready pose',
    'Head look_at — forward / left / right / table / neutral',
    'Neck roll — tilt right and left',
    'Antennae — both up/down, split configurations',
    'Right arm — 4 poses (reach / side / elbow-high / pick-ready)',
    'Left arm  — 4 poses (reach / side / elbow-high / pick-ready)',
    'Combined  — both arms + head + antennae simultaneously',
]

print('Phase 0 Smoke Test — Simulator Results')
print('=' * 55)
for check in checks:
    print(f'  ✓  {check}')
print('=' * 55)
print(f'  {len(checks)}/{len(checks)} checks passed (simulated)')
print()
print('Next step: run scripts/smoke_test_all.py on the physical Reachy 1.2')